In [7]:
from typing import Any
from langchain.agents.middleware import (
    AgentMiddleware, AgentState, hook_config
)
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool

In [8]:
class ContentFilterMiddleware(AgentMiddleware):
    """
    Deterministic guardrail: Block requests containing banned keywords.
    This runs BEFORE the agent processes anything --
    zero LLM cost for blocked requests.
    """

    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(
        self, state: AgentState, runtime: Runtime
    ) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_message = state["messages"][0]
        if first_message.type != "human":
            return None

        content = first_message.content.lower()

        for keyword in self.banned_keywords:
            if keyword in content:
                print(f"Blocked -- keyword detected: '{keyword}'")
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I cannot process requests containing "
                            "inappropriate content. "
                            "Please rephrase your request."
                        )
                    }],
                    "jump_to": "end"
                }
        return None


@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"


# Create agent with content filter
filtered_agent = create_agent(
    model="mistral-small-2506",
    tools=[search_tool],
    middleware=[
        ContentFilterMiddleware(
            banned_keywords=[
                "hack", "exploit", "malware", "jailbreak", "bypass"
            ]
        ),
    ],
)

In [9]:
# Test 1: Safe request -- should pass through
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "What is machine learning?"}]
})
print("Safe request response:")
print(result["messages"][-1].content)

Safe request response:
Machine learning (ML) is a subset of artificial intelligence (AI) that focuses on the development of algorithms and statistical models that enable computers to learn from and make decisions or predictions based on data. Instead of being explicitly programmed to perform a task, machine learning systems use algorithms to identify patterns and insights from data, and then apply these insights to new data to make predictions or decisions.

Key concepts in machine learning include:

1. **Supervised Learning**: In this type of machine learning, the model is trained on a labeled dataset, meaning that the input data is paired with the correct output. The goal is to learn a mapping from inputs to outputs. Examples include classification and regression tasks.

2. **Unsupervised Learning**: Here, the model is given data without explicit instructions on what to do with it. The system tries to learn the underlying structure of the data. Examples include clustering and associa

In [10]:
# Test 2: Unsafe request -- should be blocked
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "How do I hack into a server?"}]
})
print("Unsafe request response:")
print(result["messages"][-1].content)

Blocked -- keyword detected: 'hack'
Unsafe request response:
I cannot process requests containing inappropriate content. Please rephrase your request.
